# Replication: *SOTA or Luck? The Winner's Curse in LLM Leaderboards*

This notebook regenerates every result in the paper from the data in `resources/`:

1. It runs the seven experiment scripts below, which write their results to `results/` (the committed copies are the ones the paper reports, so a rerun should leave `git diff results/` empty).
2. Each section then reads one experiment's results, shows them, and writes the paper's tables (`.tex`) and figures (`.pdf`, `.png`) to `notebooks/output/`.
3. The last section recomputes every number quoted in the paper's text. If you set `PAPER_TEX` to the paper's LaTeX source, it also checks that each one appears there verbatim.

Run it from the repository root or from `notebooks/` after `uv sync --all-groups`.

| Script | Results | Paper tables / figures |
|---|---|---|
| `scripts/run_ruler_audit.py` | `results/ruler_audit/` | Fig. 1 (`ruler_verified_headline`), `benchmark_summary`, `ruler_verified`, `alpha_sensitivity`, `sizing`, RPM figures (and `results/ruler_audit/ruler-report.html`, not part of the paper) |
| `scripts/run_dependence.py` | `results/dependence/` | `dependence`, `dependence_ranks` |
| `scripts/run_sample_splitting.py` | `results/sample_splitting/` | `split_aggregate`, `verified_split`, `all_splits` |
| `scripts/run_calibrated_simulation.py` | `results/calibrated_simulation/` | `calibrated_main`, `calibrated_comparison`, `calibrated_bias`, `calibrated_rmse_diff`, `calibrated_full` |
| `scripts/run_gamma_sensitivity.py` | `results/gamma_sensitivity/` | `gamma_bias`, `gamma_m` |
| `scripts/run_gamma_table.py` | `results/gamma_table/` | `gamma_table_tied`, `gamma_table_close`, `gamma_table_spread` |
| `scripts/run_two_model_toy.py` | `results/two_model_toy/` | `two_model_toy_revision` figure, appendix toy numbers |

The other scripts together take about 3 minutes, and `run_gamma_table.py` about 5 minutes on 10 cores. Set `RUN_EXPERIMENTS = False` below to build the tables and figures from the committed results instead.

In [1]:
import json
import os
import re
import subprocess
import sys
from functools import partial
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import binom

from llm_wc.benchmarks import SETTINGS, SHORT_LABELS, SWE_SNAPSHOT
from llm_wc.wc.ruler_figures import plot_ruler_headline, plot_rpm_panels, plot_rpm_with_tiers, plot_two_model_toy
from llm_wc.wc.ruler_report import load_audit, write_ruler_report

REPO = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RESULTS = REPO / "results"
OUT = REPO / "notebooks" / "output"
TABLES, FIGURES = OUT / "tables", OUT / "figures"
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
read = partial(pd.read_csv, float_precision="round_trip")  # exact float round trip
# The split and gamma runs list the LiveBench categories alphabetically; their tables keep that order.
ALPHA_ORDER = [*SETTINGS[:-2], "livebench_instruction_following", "livebench_language"]


def esc(s):
    return str(s).replace("&", r"\&").replace("_", r"\_").replace("%", r"\%").replace("\u2013", "--")


def tabular(cols, header, rows):
    body = "\n".join(" & ".join(map(str, r)) + r" \\" for r in rows)
    return f"\\begin{{tabular}}{{@{{}}{cols}@{{}}}}\n\\toprule\n{header} \\\\\n\\midrule\n{body}\n\\bottomrule\n\\end{{tabular}}"


def write_table(name, tex):
    (TABLES / f"{name}.tex").write_text(tex + "\n")

## 0. Run the experiments

Each script writes `results/<name>/`. This takes about 8 minutes.

In [2]:
RUN_EXPERIMENTS = True
EXPERIMENTS = ["ruler_audit", "dependence", "sample_splitting", "calibrated_simulation",
               "gamma_sensitivity", "gamma_table", "two_model_toy"]
if RUN_EXPERIMENTS:
    for name in EXPERIMENTS:
        print(f"running scripts/run_{name}.py", flush=True)
        subprocess.run([sys.executable, str(REPO / "scripts" / f"run_{name}.py")], cwd=REPO, check=True,
                       stdout=subprocess.DEVNULL)

running scripts/run_ruler_audit.py


running scripts/run_dependence.py


running scripts/run_sample_splitting.py


running scripts/run_calibrated_simulation.py


running scripts/run_gamma_sensitivity.py


running scripts/run_gamma_table.py


running scripts/run_two_model_toy.py


## 1. RULER audit of 14 settings

`m = floor(N^0.9)`, B = 5000, seed 42. `summary.csv` has one row per setting. Each setting folder holds `models.csv` (raw score order, with tie-averaged displayed adjustments, 80/90/95/99% rank-mass intervals and tiers), `position_bias.csv`, `rpm.csv` and `tiers.csv`.

In [3]:
audit = read(RESULTS / "ruler_audit" / "summary.csv").set_index("setting")
A = {k: load_audit(RESULTS / "ruler_audit", k) for k in SETTINGS}
audit

,label,n_tasks,n_models,m,n_bootstraps,seed,position_1_adjustment_pp,leader_adjustment_pp,leader_adjustment_low_pp,leader_adjustment_high_pp,leader_ordinary_adjustment_pp,leader_p_rank_1,tiers_80,tier_1_size_80,tiers_90,tier_1_size_90,tiers_95,tier_1_size_95,tiers_99,tier_1_size_99
setting,,,,,,,,,,,,,,,,,,,,
swebench_bash_only,SWE Bash-only,500,41,268,5000,42,1.093510,1.093510,1.093510,1.093510,0.961920,0.416500,10,6,9,9,7,13,6,16
swebench_verified,SWE Verified,500,173,268,5000,42,1.122807,0.896339,0.669872,1.122807,0.672440,0.304783,15,13,13,15,11,24,8,29
swebench_multimodal,SWE Multimodal,510,12,273,5000,42,0.940765,0.625966,0.311168,0.940765,0.543157,0.347817,3,5,3,5,2,8,2,10
swebench_multilingual,SWE Multilingual,300,14,169,5000,42,1.499504,1.499504,1.499504,1.499504,1.240533,0.448433,4,9,3,11,3,12,3,12
swebench_pro,SWE Pro,395,14,217,5000,42,0.872294,0.872294,0.872294,0.872294,0.618430,0.748667,6,4,6,4,4,5,4,5
tau2_airline,Tau2 Airline,50,7,33,5000,42,3.067726,3.067726,3.067726,3.067726,2.123200,0.745417,2,5,1,7,1,7,1,7
tau2_retail,Tau2 Retail,114,7,70,5000,42,2.200237,2.200237,2.200237,2.200237,1.936930,0.630833,2,5,2,6,2,6,2,6
tau2_telecom,Tau2 Telecom,114,7,70,5000,42,0.175408,0.175408,0.175408,0.175408,0.069254,0.943400,6,1,4,2,3,3,2,3
alpaca_eval_2,AlpacaEval 2,805,208,412,5000,42,0.532094,0.532094,0.532094,0.532094,0.455915,0.534200,21,3,16,4,14,4,11,5


In [4]:
# benchmark_summary: full audit
# Position 1 scores: the leader's raw score and that score minus its tie-averaged adjustment
leader_raw = {k: A[k]["models"].raw_score_pct[0] for k in audit.index}
write_table("benchmark_summary", tabular(
    "lcccccc", "Benchmark & \\shortstack{Number\\\\of Tasks} & \\shortstack{Number\\\\of Models} & \\shortstack{Position 1\\\\Score (Raw)} & "
    "\\shortstack{Position 1\\\\Score (Adjusted)} & \\shortstack{Number\\\\of Tiers} & \\shortstack{Number of\\\\Models in Tier 1}",
    [[SHORT_LABELS[k], r.n_tasks, r.n_models, f"{leader_raw[k]:.2f}", f"{leader_raw[k] - r.leader_adjustment_pp:.2f}",
      r.tiers_80, r.tier_1_size_80] for k, r in audit.iterrows()]))

# alpha_sensitivity: tier count / Tier 1 size at 80/90/95/99% rank mass
LEVELS = ("80", "90", "95", "99")
write_table("alpha_sensitivity", tabular(
    "l*{8}{>{\\centering\\arraybackslash}p{1.1cm}}",
    "& \\multicolumn{4}{c}{Number of Tiers} & \\multicolumn{4}{c}{Number of Models in Tier 1} \\\\\n"
    "\\cmidrule(lr){2-5}\\cmidrule(l){6-9}\n"
    "Benchmark & 80\\% & 90\\% & 95\\% & 99\\% & 80\\% & 90\\% & 95\\% & 99\\%",
    [[SHORT_LABELS[k]] + [r[f"tiers_{l}"] for l in LEVELS] + [r[f"tier_1_size_{l}"] for l in LEVELS]
     for k, r in audit.iterrows()]))

In [5]:
# ruler_verified: RULER for the first seven raw-order models on SWE-bench Verified (keeps the 5-7 tie intact)
def short_name(name):
    name = name.replace(" (20251101)", "").replace(" (20250929)", "")
    for a, b in [("live-SWE-agent", "live-SWE"), ("Sonar Foundation Agent", "Sonar"), (" medium", " med."),
                 (" Preview", ""), ("EPAM AI/Run Developer Agent v20250719", "EPAM AI/Run v20250719"),
                 (" (high reasoning)", " (high)"), (" (2025-11-18)", ""), (" (2025-09-02)", "")]:
        name = name.replace(a, b)
    return name

verified = A["swebench_verified"]["models"]
write_table("ruler_verified", tabular(
    "c>{\\raggedright\\arraybackslash}p{4.7cm}cccc",
    "\\shortstack{Raw\\\\Rank} & Model & \\shortstack{Raw Score\\\\(\\%)} & \\shortstack{Score Correction\\\\(pp)} & "
    "\\shortstack{80\\% Rank\\\\Interval} & Tier",
    [[esc(r["rank"]), esc(short_name(r.model)), f"{r.raw_score_pct:.2f}", f"{r.adjustment_pp:.2f}",
      f"[{r.rank_low_80},{r.rank_high_80}]", r.tier_80] for _, r in verified.head(7).iterrows()]))
verified.head(10)

,position,model_id,model,raw_score_pct,rank,rank_start,rank_end,adjustment_pp,adjustment_low_pp,adjustment_high_pp,...,tier_80,rank_low_90,rank_high_90,tier_90,rank_low_95,rank_high_95,tier_95,rank_low_99,rank_high_99,tier_99
0,1,live-swe-agent-claude-4-5-opus-medium-20251101,live-SWE-agent + Claude 4.5 Opus medium (20251...,79.2,1–2,1,2,0.896339,0.669872,1.122807,...,1,1,7,1,1,9,1,1,13,1
1,2,sonar-foundation-agent-claude-4-5-opus,Sonar Foundation Agent + Claude 4.5 Opus,79.2,1–2,1,2,0.896339,0.669872,1.122807,...,1,1,7,1,1,9,1,1,13,1
2,3,trae-doubao-seed-code,TRAE + Doubao-Seed-Code,78.8,3,3,3,0.598772,0.598772,0.598772,...,1,1,9,1,1,12,1,1,17,1
3,4,live-swe-agent-gemini-3-pro-preview-2025-11-18,live-SWE-agent + Gemini 3 Pro Preview (2025-11...,77.4,4,4,4,0.693490,0.693490,0.693490,...,1,2,16,1,1,18,1,1,23,1
4,5,atlassian-rovo-dev-2025-09-02,Atlassian Rovo Dev (2025-09-02),76.8,5–7,5,7,0.625300,0.590464,0.664291,...,1,2,17,1,2,19,1,1,24,1
5,6,epam-ai-run-developer-agent-v20250719-claude-4...,EPAM AI/Run Developer Agent v20250719 + Claude...,76.8,5–7,5,7,0.625300,0.590464,0.664291,...,1,2,18,1,1,21,1,1,25,1
6,7,mini-swe-agent-claude-4-5-opus-high-reasoning,mini-SWE-agent + Claude 4.5 Opus (high reasoning),76.8,5–7,5,7,0.625300,0.590464,0.664291,...,1,3,19,1,2,21,1,1,25,1
7,8,acoder,ACoder,76.4,8,8,8,0.535883,0.535883,0.535883,...,1,3,21,1,2,23,1,1,28,1
8,9,mini-swe-agent-gemini-3-flash-high-reasoning,mini-SWE-agent + Gemini 3 Flash (high reasoning),75.8,9–10,9,10,0.428979,0.419028,0.438930,...,1,4,24,1,3,26,1,1,31,1
9,10,mini-swe-agent-minimax-m2-5-high-reasoning,mini-SWE-agent + MiniMax M2.5 (high reasoning),75.8,9–10,9,10,0.428979,0.419028,0.438930,...,1,4,24,1,3,27,1,1,32,1


In [6]:
# sizing: task-pool planning, N >= (delta / eps)^2 with delta = sqrt(500) * b_500(1) on Verified
delta = A["swebench_verified"]["position_bias"].adjustment_pp[0] * np.sqrt(500)
write_table("sizing", tabular("cc", "\\shortstack{Target Expected\\\\Winner Bias (pp)} & \\shortstack{Approximate Number\\\\of Tasks}",
                              [[f"{x:.2f}", f"{int(np.ceil((delta / x) ** 2)):,}"] for x in [2, 1, .5, .25, .1]]))
delta

np.float64(25.106719982331104)

In [7]:
# Figure 1 and the RPM figures
plot_ruler_headline(verified, A["swebench_verified"]["rpm"],
                    title="SWE-Bench Verified: Score Overstatement and Rank Uncertainty",
                    tier_1_size=int(audit.loc["swebench_verified", "tier_1_size_80"]),
                    output_path=FIGURES / "ruler_verified_headline.pdf")
plot_rpm_with_tiers(A["swebench_verified"]["rpm"], verified.tier_80.to_numpy(), top=30, vmax=.35, figsize=(6.2, 5.4),
                    title="SWE-bench Verified: top 30 of 173 models", output_path=FIGURES / "rpm_verified_revision.pdf")
report_labels = {s["key"]: s["label"] for s in json.loads((REPO / "resources/ruler_report/settings.json").read_text())["settings"]}
plot_rpm_panels([(A[k]["rpm"], A[k]["models"].tier_80.to_numpy(), list(A[k]["models"].model), report_labels[k])
                 for k in ["tau2_airline", "tau2_retail", "tau2_telecom"]],
                output_path=FIGURES / "tau2_rpm_revision.pdf")

In [8]:
# Interactive report covering every model in all 14 settings
report = write_ruler_report(RESULTS / "ruler_audit", RESULTS / "ruler_audit" / "ruler-report.html")

## 2. Task dependence

Pooled rows, rows stratified by repository, and a repository wild-cluster bootstrap (Webb weights) for the five SWE-bench variants, plus Tau2 telecom's three subcategories.

In [9]:
dep = read(RESULTS / "dependence" / "summary.csv")
d = dep.set_index(["setting", "scheme"])
swe = [k for k in SETTINGS if k.startswith("swebench_")]
SCHEMES = {"pooled": "Pooled", "stratified": "Stratified", "wild": "Cluster"}  # "wild" is the cluster stress test
write_table("dependence", tabular(
    "lc*{6}{>{\\centering\\arraybackslash}p{1.4cm}}",
    "& & \\multicolumn{3}{c}{Position 1 Correction (pp)} & \\multicolumn{3}{c}{Number of Models in Tier 1} \\\\\n"
    "\\cmidrule(lr){3-5}\\cmidrule(l){6-8}\n"
    "Benchmark & \\shortstack{Number of\\\\Repositories} & Pooled & Stratified & Cluster & Pooled & Stratified & Cluster",
    [[SHORT_LABELS[k], d.loc[(k, "pooled"), "clusters"],
      *[f"{d.loc[(k, s), 'position_1_adjustment_pp']:.2f}" for s in SCHEMES],
      *[int(d.loc[(k, s), "tier_1_size_80"]) for s in SCHEMES]] for k in swe]))
write_table("dependence_ranks", tabular(
    "llccccc",
    "& & \\multicolumn{3}{c}{Rank-1 Probability (\\%)} & & \\\\\n\\cmidrule(lr){3-5}\n"
    "Benchmark & \\shortstack[l]{Resampling\\\\Scheme} & Model 1 & Model 2 & Model 3 & \\shortstack{Number\\\\of Tiers} & "
    "\\shortstack{Number of\\\\Models in Tier 1}",
    [[SHORT_LABELS[k], label, *[f"{100 * d.loc[(k, s), f'p_rank_1_position_{i}']:.1f}" for i in (1, 2, 3)],
      int(d.loc[(k, s), "tiers_80"]), int(d.loc[(k, s), "tier_1_size_80"])]
     for k in swe for s, label in SCHEMES.items()]))
dep

,setting,scheme,n_tasks,clusters,resample_size,position_1_adjustment_pp,p_rank_1_position_1,p_rank_1_position_2,p_rank_1_position_3,tiers_80,tier_1_size_80
0,swebench_bash_only,pooled,500,12,268,1.093510,0.416500,0.168483,0.183733,10,6
1,swebench_bash_only,stratified,500,12,268,1.205371,0.431017,0.155400,0.180783,10,6
2,swebench_bash_only,wild,500,12,500,0.860774,0.574400,0.181000,0.158800,12,4
3,swebench_verified,pooled,500,12,268,1.122807,0.304783,0.290217,0.263967,15,13
4,swebench_verified,stratified,500,12,268,1.201952,0.289657,0.293557,0.261957,15,12
5,swebench_verified,wild,500,12,500,1.158379,0.305500,0.290900,0.338200,24,4
6,swebench_multimodal,pooled,510,12,273,0.940765,0.347817,0.339583,0.220150,3,5
7,swebench_multimodal,stratified,510,12,273,0.820555,0.347650,0.343333,0.221983,3,5
8,swebench_multimodal,wild,510,12,510,0.843119,0.318300,0.375500,0.270000,3,5
9,swebench_multilingual,pooled,300,41,169,1.499504,0.448433,0.335817,0.091233,4,9


## 3. Half-split validation

For each setting, 200 random half-splits. Half A selects the model at positions 1–5 and estimates the adjustment. Half B scores the same models. `split_error` is score_A − score_B; `residual` is split_error − adjustment. Standard errors describe variability over partitions of the same task pool.

In [10]:
splits = read(RESULTS / "sample_splitting" / "splits.csv")
rows = []
for k in ALPHA_ORDER:
    g = splits[splits.setting == k].sort_values(["split", "position"])
    shape = (g.split.nunique(), g.position.nunique())
    err, adj = g.split_error.to_numpy().reshape(shape), g.adjustment.to_numpy().reshape(shape)
    res = err - adj
    for p in range(shape[1]):
        rows.append({"setting": k, "position": p + 1,
                     "split_error_pp": 100 * err.mean(0)[p], "split_error_se_pp": 100 * (err.std(0) / np.sqrt(shape[0]))[p],
                     "adjustment_pp": 100 * adj.mean(0)[p], "adjustment_se_pp": 100 * (adj.std(0) / np.sqrt(shape[0]))[p],
                     "residual_pp": 100 * res.mean(0)[p], "residual_se_pp": 100 * (res.std(0) / np.sqrt(shape[0]))[p]})
per_position = pd.DataFrame(rows)

agg = per_position.groupby("position").agg(
    raw=("split_error_pp", lambda x: x.abs().mean()), adjusted=("residual_pp", lambda x: x.abs().mean()))
agg["reduction_pct"] = 100 * (1 - agg.adjusted / agg.raw)
write_table("split_aggregate", tabular(
    "cccc",
    "& \\multicolumn{3}{c}{Magnitude of Mean Error} \\\\\n\\cmidrule(l){2-4}\n"
    "Rank & Before RULER & After RULER & Reduction (\\%)",
    [[k, f"{r.raw:.2f}", f"{r.adjusted:.2f}", f"{r.reduction_pct:.1f}"]
     for k, r in agg.iterrows()]))

v = per_position[per_position.setting == "swebench_verified"]
write_table("verified_split", tabular("cccc",
    "Rank & \\shortstack{Mean Error\\\\Before RULER} & \\shortstack{Mean Score\\\\Correction} & \\shortstack{Mean Error\\\\After RULER}",
    [[r.position, f"{r.split_error_pp:.2f}", f"{r.adjustment_pp:.2f}", f"{r.residual_pp:+.2f}"] for _, r in v.iterrows()]))

head = (r"& & \multicolumn{3}{c}{Mean $\pm$ SE Across Splits} \\" "\n"
        r"\cmidrule(l){3-5}" "\n"
        r"Benchmark & Rank & \shortstack{Error Before\\RULER} & \shortstack{Score\\Correction} & \shortstack{Error After\\RULER} \\")
body = "\n".join(" & ".join(map(str, [
    SHORT_LABELS[r.setting], r.position, f"${r.split_error_pp:+.2f}\\pm{r.split_error_se_pp:.2f}$",
    f"${r.adjustment_pp:+.2f}\\pm{r.adjustment_se_pp:.2f}$", f"${r.residual_pp:+.2f}\\pm{r.residual_se_pp:.2f}$"])) + r" \\" for _, r in per_position.iterrows())
write_table("all_splits", (
    r"\begin{longtable}{@{}lcccc@{}}" + "\n"
    + r"\caption{The reduction in selection error varies across benchmarks and ranks.}\label{tab:all_splits}\\"
    + "\n\\toprule\n" + head + "\n\\midrule\n\\endfirsthead\n\\toprule\n" + head + "\n\\midrule\n\\endhead\n\\bottomrule\n\\endfoot\n"
    + body + "\n\\end{longtable}\n"
    + r"\floatnote{All 70 benchmark--position combinations use 200 random half-splits. Selection halves contain $N_S=\lfloor N/2\rfloor$ "
    r"tasks, with $m=\lfloor N_S^{0.9}\rfloor$ and $B=2{,}000$. Error Before RULER is the selected model's selection-half score minus "
    r"its evaluation-half score. Score Correction is the estimated bias. Error After RULER subtracts that correction within each split. "
    r"Entries are means and standard errors (SEs) across splits. All scores and errors are in pp. SEs measure Monte Carlo precision over "
    r"partitions of the same frozen task pool, conditional on that pool.}"))
agg

,raw,adjusted,reduction_pct
position,,,
1,2.381706,0.727410,69.458466
2,1.427186,0.742114,48.001626
3,1.339436,0.710889,46.926238
4,1.101446,0.496105,54.958733
5,1.231587,0.582987,52.663767


## 4. Calibrated simulation

A logistic DGP is fitted to SWE-bench Verified (N = 472 nonconstant tasks, J = 173). Over 100 synthetic benchmarks, the selected models' score errors at positions 1–5 are compared with three adjustments computed on the same draw: the m-out-of-N bootstrap (m = 255), the ordinary bootstrap, and beta-binomial empirical Bayes.

In [11]:
reps = read(RESULTS / "calibrated_simulation" / "replications.csv")
M = lambda col: reps.pivot(index="replication", columns="position", values=col).to_numpy()
raw = M("raw_error")
n_rep = len(raw)
se = lambda x: 100 * x.std(0, ddof=1) / np.sqrt(n_rep)
true_bias, true_bias_se = 100 * raw.mean(0), se(raw)
def rmse(e):  # RMSE and per-replication delta-method influence values, so se(psi) is its SE and se(psi_a - psi_b) a paired SE
    r = np.sqrt((e ** 2).mean(0))
    return r, (e ** 2 - (e ** 2).mean(0)) / (2 * r)
raw_rmse, raw_psi = rmse(raw)
raw_rmse_se = se(raw_psi)
moon_rmse, moon_psi = rmse(raw - M("moon"))
METHODS = ["moon", "ordinary", "empirical_bayes", "sample_splitting"]
methods = {}
for name in METHODS:
    res = raw - M(name)
    r, psi = rmse(res)
    methods[name] = pd.DataFrame({
        "adjustment_pp": 100 * M(name).mean(0), "residual_pp": 100 * res.mean(0), "residual_se_pp": se(res),
        "bias_removed_pct": 100 * (1 - res.mean(0) / raw.mean(0)), "rmse_pp": 100 * r,
        "rmse_se_pp": se(psi), "rmse_gap_pp": 100 * (r - moon_rmse), "rmse_gap_se_pp": se(psi - moon_psi)},
        index=pd.Index(range(1, raw.shape[1] + 1), name="position"))
raw_gap_pp, raw_gap_se = 100 * (raw_rmse - moon_rmse), se(raw_psi - moon_psi)

pm = lambda v, s, sign="": f"${v:{sign}.2f}\\pm{s:.2f}$"
head = ("Rank & No Correction & \\shortstack{$m$-out-of-$N$\\\\bootstrap} & \\shortstack{Standard\\\\bootstrap} & EB & "
        "\\shortstack{Sample\\\\splitting}")
mo = methods["moon"]
write_table("calibrated_main", tabular(
    "ccccc", "& \\multicolumn{2}{c}{Mean Signed Error (pp)} & \\multicolumn{2}{c}{RMSE (pp)} \\\\\n"
    "\\cmidrule(lr){2-3}\\cmidrule(l){4-5}\n"
    "Rank & No Correction & \\shortstack{$m$-out-of-$N$\\\\bootstrap} & No Correction & \\shortstack{$m$-out-of-$N$\\\\bootstrap}",
    [[k + 1, pm(true_bias[k], true_bias_se[k], "+"), pm(mo.residual_pp.iloc[k], mo.residual_se_pp.iloc[k], "+"),
      pm(raw_rmse[k] * 100, raw_rmse_se[k]), pm(mo.rmse_pp.iloc[k], mo.rmse_se_pp.iloc[k])] for k in range(5)]))
write_table("calibrated_comparison", tabular(
    "cccccc", "& \\multicolumn{5}{c}{RMSE (pp)} \\\\\n\\cmidrule(l){2-6}\n" + head,
    [[k + 1, pm(raw_rmse[k] * 100, raw_rmse_se[k])] + [pm(methods[x].rmse_pp.iloc[k], methods[x].rmse_se_pp.iloc[k]) for x in METHODS]
     for k in range(5)]))
write_table("calibrated_bias", tabular(
    "cccccc", "& \\multicolumn{5}{c}{Mean Signed Error (pp)} \\\\\n\\cmidrule(l){2-6}\n" + head,
    [[k + 1, pm(true_bias[k], true_bias_se[k], "+")]
     + [pm(methods[x].residual_pp.iloc[k], methods[x].residual_se_pp.iloc[k], "+") for x in METHODS] for k in range(5)]))
write_table("calibrated_rmse_diff", tabular(
    "ccccc", "& \\multicolumn{4}{c}{RMSE minus $m$-out-of-$N$ RMSE (pp)} \\\\\n\\cmidrule(l){2-5}\n"
    "Rank & No Correction & \\shortstack{Standard\\\\bootstrap} & EB & \\shortstack{Sample\\\\splitting}",
    [[k + 1, pm(raw_gap_pp[k], raw_gap_se[k], "+")]
     + [pm(methods[x].rmse_gap_pp.iloc[k], methods[x].rmse_gap_se_pp.iloc[k], "+") for x in METHODS[1:]] for k in range(5)]))
names = {"moon": "$m$-out-of-$N$ bootstrap", "ordinary": "Standard bootstrap", "empirical_bayes": "Beta-binomial EB",
         "sample_splitting": "Sample splitting"}
write_table("calibrated_full", tabular(
    "clccc",
    "& & & \\multicolumn{2}{c}{After Correction} \\\\\n\\cmidrule(l){4-5}\n"
    "Rank & Correction Method & \\shortstack{Mean Score\\\\Correction (pp)} & \\shortstack{Mean Residual\\\\(pp)} & RMSE (pp)",
    [[k + 1, names[x], f"{methods[x].adjustment_pp.iloc[k]:.2f}", f"{methods[x].residual_pp.iloc[k]:+.2f}",
      f"{methods[x].rmse_pp.iloc[k]:.2f}"] for k in range(5) for x in methods]))
calibration = np.load(RESULTS / "calibrated_simulation" / "calibration.npz")
pd.concat(methods, axis=1).round(3)

moon                                                      \
         adjustment_pp residual_pp residual_se_pp bias_removed_pct rmse_pp   
position                                                                     
1                1.203       0.209          0.154           85.213   1.551   
2                0.860      -0.061          0.175          107.574   1.738   
3                0.778      -0.120          0.179          118.235   1.784   
4                0.753       0.230          0.164           76.637   1.644   
5                0.724       0.241          0.179           75.040   1.795   

                                                    ordinary              ...  \
         rmse_se_pp rmse_gap_pp rmse_gap_se_pp adjustment_pp residual_pp  ...   
position                                                                  ...   
1             0.099         0.0            0.0         0.953       0.458  ...   
2             0.142         0.0            0.0         0.602       0.197  ...   
3             0.142         0.0            0.0         0.527       0.131  ...   
4             0.127         0.0            0.0         0.571       0.412  ...   
5             0.133         0.0            0.0         0.565       0.400  ...   

         empirical_bayes                sample_splitting              \
             rmse_gap_pp rmse_gap_se_pp    adjustment_pp residual_pp   
position                                                               
1                  0.204          0.074            1.348       0.063   
2                  0.002          0.072            0.816      -0.017   
3                  0.004          0.052            0.726      -0.068   
4                  0.074          0.061            0.820       0.163   
5                  0.058          0.042            0.837       0.128   

                                                                         \
         residual_se_pp bias_removed_pct rmse_pp rmse_se_pp rmse_gap_pp   
position                                                                  
1                 0.165           95.502   1.638      0.095       0.087   
2                 0.178          102.103   1.773      0.145       0.034   
3                 0.185          110.329   1.839      0.147       0.055   
4                 0.173           83.384   1.731      0.137       0.087   
5                 0.188           86.746   1.872      0.137       0.077   

                         
         rmse_gap_se_pp  
position                 
1                 0.032  
2                 0.031  
3                 0.023  
4                 0.032  
5                 0.029  

[5 rows x 32 columns]

## 5. Sensitivity to the resample exponent γ

m = max(2, floor(N^γ)); γ = 1 is the ordinary bootstrap. The paper tables show γ from 0.4 to 0.9.

In [12]:
gamma = read(RESULTS / "gamma_sensitivity" / "summary.csv")
gamma_range = gamma[gamma.gamma.between(0.4, 0.9)]  # exponents reported in the robustness appendix
gs = sorted(gamma_range.gamma.unique())
by = {k: g.sort_values("gamma") for k, g in gamma_range.groupby("setting")}
header = (f"& \\multicolumn{{{len(gs)}}}{{c}}{{Resample-Size Exponent $\\gamma$}} \\\\\n\\cmidrule(l){{2-{len(gs) + 1}}}\n"
          "Benchmark & " + " & ".join(f"{g:g}" for g in gs))
gcols = f"l*{{{len(gs)}}}{{>{{\\centering\\arraybackslash}}p{{0.7cm}}}}"
write_table("gamma_bias", tabular(gcols, header,
    [[SHORT_LABELS[k]] + [f"{x:.2f}" for x in by[k].position_1_adjustment_pp] for k in ALPHA_ORDER]))
write_table("gamma_m", tabular(gcols, header,
    [[SHORT_LABELS[k]] + [str(x) for x in by[k].m] for k in ALPHA_ORDER]))
gamma.pivot(index="setting", columns="gamma", values="position_1_adjustment_pp").loc[ALPHA_ORDER].round(2)

gamma,0.10,0.20,0.30,0.40,0.50,0.60,0.70,0.80,0.90,0.95,1.00
setting,,,,,,,,,,,
swebench_bash_only,0.91,1.08,1.43,1.61,1.59,1.53,1.38,1.26,1.09,1.04,0.96
swebench_verified,1.12,1.34,1.72,1.91,1.90,1.77,1.56,1.34,1.12,1.04,0.97
swebench_multimodal,0.78,0.88,1.03,1.13,1.13,1.13,1.04,1.02,0.94,0.89,0.87
swebench_multilingual,1.36,1.60,1.88,2.10,2.16,2.15,1.94,1.74,1.50,1.36,1.24
swebench_pro,1.87,2.01,2.14,2.10,1.90,1.67,1.43,1.16,0.87,0.73,0.62
tau2_airline,6.11,6.11,6.17,6.13,5.75,5.40,4.80,4.00,3.07,2.60,2.12
tau2_retail,2.55,2.55,2.94,3.01,2.93,2.78,2.60,2.45,2.20,2.08,1.94
tau2_telecom,1.43,1.43,1.14,1.23,1.23,0.93,0.58,0.36,0.18,0.13,0.07
alpaca_eval_2,1.40,1.46,1.47,1.33,1.13,0.93,0.77,0.64,0.53,0.49,0.46


## 6. Lookup table for the resample exponent

J models with independent Bernoulli outcomes on N tasks; true means evenly spaced over k·σ around p, σ = sqrt(p(1−p)/N), for tied (k = 0), close (k = 2) and spread (k = 15). 1000 synthetic benchmarks, B = 2000. RMSE over all J positions of corrected score minus the true mean of the model observed there.

In [13]:
# gamma_table_{tied,close,spread}: RMSE over all positions; in each (N, p, J) column the three
# smallest distinct two-decimal values get three shades
gt = read(RESULTS / "gamma_table" / "summary.csv").assign(err_pp=lambda t: 100 * t.rmse)
gt_g = gt[gt.gamma.notna()]
SHADES = {1: r"\cellcolor[HTML]{0B5D1E}\color{white}\bfseries ", 2: r"\cellcolor[HTML]{2E8B3E}\color{white}\bfseries ",
          3: r"\cellcolor[HTML]{CDEFD4}\bfseries "}
gt_j, gt_p = sorted(gt.n_models.unique()), sorted(gt.p.unique())
ncol = len(gt_p) * len(gt_j)
for spread in ["tied", "close", "spread"]:
    header = (" & " + " & ".join(f"\\multicolumn{{{len(gt_j)}}}{{c}}{{$p={p:g}$}}" for p in gt_p) + " \\\\\n"
              + "".join(f"\\cmidrule(lr){{{2 + i * len(gt_j)}-{1 + (i + 1) * len(gt_j)}}}" for i in range(len(gt_p)))
              + "\n$\\gamma\\,/\\,J$ & " + " & ".join(f"{j}" for _ in gt_p for j in gt_j))
    lines = []
    for i, n in enumerate(sorted(gt.n_tasks.unique())):
        t = gt_g[(gt_g.spread == spread) & (gt_g.n_tasks == n)].pivot(index="gamma", columns=["p", "n_models"], values="err_pp").round(2)
        rank = t.rank(method="dense")
        lines += ["\\midrule"] * bool(i) + [f"\\multicolumn{{{ncol + 1}}}{{@{{}}l}}{{$N={n:,}$ tasks}} \\\\".replace(",", "{,}")]
        lines += [" & ".join([f"{g:g}"] + [SHADES.get(rank.at[g, c], "") + f"{t.at[g, c]:.2f}" for c in t.columns]) + " \\\\" for g in t.index]
    write_table(f"gamma_table_{spread}", f"\\begin{{tabular}}{{@{{}}c*{{{ncol}}}{{c}}@{{}}}}\n"
                f"\\toprule\n{header} \\\\\n\\midrule\n" + "\n".join(lines) + "\n\\bottomrule\n\\end{tabular}")

# Errors relative to the most accurate exponent in each (spread, N, p, J) cell, in %
KEY = ["spread", "n_tasks", "p", "n_models"]
best = gt_g.groupby(KEY).err_pp.min()
excess = lambda s: 100 * (s.set_index(KEY).err_pp / best - 1)
ex = {g: excess(gt_g[gt_g.gamma == g]) for g in [0.9, 1.0]} | {"range": excess(gt_g[gt_g.gamma.between(0.4, 0.9)])} | {"none": excess(gt[gt.gamma.isna()])}
r2 = gt_g.assign(err2=gt_g.err_pp.round(2))
best_set = r2[r2.err2 == r2.groupby(KEY).err2.transform("min")].groupby(KEY).gamma  # exponents tied for the lowest displayed RMSE
J = lambda s: s.index.get_level_values("n_models")
rng = lambda s: (s.min(), s.max())
gsum = {}
for sp in ["tied", "close", "spread"]:
    e9, e1, en = (ex[k].xs(sp, level="spread") for k in [0.9, 1.0, "none"])
    lo, hi = best_set.min().xs(sp, level="spread"), best_set.max().xs(sp, level="spread")
    gsum[sp] = dict(best=(lo.min(), hi.max()), best100=(lo[J(lo) == 100].min(), hi[J(hi) == 100].max()),
                    best20=(lo[J(lo) >= 20].min(), hi[J(hi) >= 20].max()),
                    e9=rng(e9), e9_10=rng(e9[J(e9) <= 10]), e9_100=rng(e9[J(e9) == 100]), e9_20=rng(e9[J(e9) >= 20]),
                    e1=rng(e1), en=rng(en), en5=rng(en[J(en) >= 5]), en2=rng(en[J(en) == 2]),
                    en5_20=rng(en[(J(en) >= 5) & (J(en) <= 20)]), en_rest=en[J(en).isin([2, 50, 100])].abs().max())
# For each spacing and J, the shaded (three smallest) exponents overlap across all (N, p) panels
r2["rank"] = r2.groupby(KEY).err2.rank(method="dense")
gt_overlap = all(set.intersection(*g.groupby(["n_tasks", "p"]).gamma.apply(set)) for _, g in r2[r2["rank"] <= 3].groupby(["spread", "n_models"]))
# Spread of the audited leaderboards in units of sigma = sqrt(pbar (1 - pbar) / N), pbar the mean score
spans = pd.DataFrame([{"all": (s.max() - s.min()) / sig, "top10": (s.max() - np.sort(s)[::-1][:10][-1]) / sig}
                      for k in SETTINGS for s in [A[k]["models"].raw_score_pct.to_numpy() / 100]
                      for sig in [np.sqrt(s.mean() * (1 - s.mean()) / audit.loc[k, "n_tasks"])]])
pd.DataFrame(gsum).map(lambda v: tuple(round(x, 2) for x in v) if isinstance(v, tuple) else round(v, 2)), spans.describe().round(1)

(                    tied           close          spread
 best          (0.1, 0.7)      (0.1, 0.8)      (0.7, 1.0)
 best100       (0.4, 0.6)      (0.6, 0.8)      (0.8, 1.0)
 best20        (0.2, 0.7)      (0.2, 0.8)      (0.7, 1.0)
 e9         (8.03, 57.13)    (1.72, 7.28)     (0.0, 4.85)
 e9_10      (8.03, 17.86)    (3.68, 7.28)     (0.0, 4.85)
 e9_100    (46.79, 57.13)    (1.72, 2.61)      (0.0, 0.0)
 e9_20     (21.01, 57.13)    (1.72, 4.13)     (0.0, 1.72)
 e1        (11.71, 110.1)   (3.62, 13.48)     (0.0, 3.96)
 en        (19.7, 496.97)  (-3.26, 88.04)   (-9.89, 3.16)
 en5      (67.05, 496.97)  (25.43, 88.04)   (-9.89, 3.16)
 en2        (19.7, 23.53)  (-3.26, -0.82)    (0.01, 0.12)
 en5_20   (67.05, 191.02)  (25.43, 61.69)  (-9.89, -3.56)
 en_rest           496.97           88.04            3.16,
         all  top10
 count  14.0   14.0
 mean   18.0    4.7
 std    13.5    3.8
 min     2.7    0.6
 25%     8.8    2.3
 50%    14.8    3.7
 75%    21.5    5.1
 max    51.4   14.3)

## 7. Two-model illustration

N = 500 tasks, success probabilities (0.500, 0.505) or (0.500, 0.600), 3000 benchmark draws, B = 2000, γ = 0.9.

In [14]:
draws = read(RESULTS / "two_model_toy" / "gamma09_draws.csv")
toy = {}
for name, g in draws.groupby("scenario", sort=False):
    p1, p2 = g.p1.iloc[0], g.p2.iloc[0]
    n, k = 500, len(g)
    x = np.arange(n + 1) / n
    mass = np.outer(binom.pmf(np.arange(n + 1), n, p1), binom.pmf(np.arange(n + 1), n, p2))
    target = np.where(x[:, None] > x[None, :], p1, np.where(x[:, None] < x[None, :], p2, (p1 + p2) / 2))
    err, res = g.raw - g.selected_true, g.adjusted - g.selected_true
    toy[name] = dict(
        raw_mean=g.raw.mean(), adjusted_mean=g.adjusted.mean(), selected_true_mean=g.selected_true.mean(),
        raw_bias_pp=100 * err.mean(), raw_bias_se_pp=100 * err.std(ddof=1) / np.sqrt(k),
        residual_pp=100 * res.mean(), residual_se_pp=100 * res.std(ddof=1) / np.sqrt(k),
        mean_adjustment_pp=100 * g.adjustment.mean(), better_model_wins=g.winner.mean(),
        exact_bias_pp=100 * np.sum(mass * (np.maximum(x[:, None], x[None, :]) - target)),
        rpm=np.mean([[[1 - q, q], [q, 1 - q]] for q in g.p_model2_rank1], axis=0))
small = draws[draws.scenario == "small_gap"]
plot_two_model_toy(small.raw.to_numpy(), small.adjusted.to_numpy(), toy["small_gap"]["selected_true_mean"],
                   toy["small_gap"]["rpm"], toy["large_gap"]["rpm"], output_path=FIGURES / "two_model_toy_revision.pdf")
toy06 = json.loads((RESULTS / "two_model_toy" / "gamma06_summary.json").read_text())["scenarios"]
pd.DataFrame(toy).drop("rpm")

,small_gap,large_gap
raw_mean,0.515111,0.599315
adjusted_mean,0.505013,0.597552
selected_true_mean,0.50283,0.599933
raw_bias_pp,1.228133,-0.061867
raw_bias_se_pp,0.034028,0.040507
residual_pp,0.218312,-0.238161
residual_se_pp,0.036832,0.043193
mean_adjustment_pp,1.009822,0.176295
better_model_wins,0.566,0.999333
exact_bias_pp,1.245567,0.007831


## 8. Numbers quoted in the paper

Each claim below is recomputed from the results above and formatted as in the text. Set the environment variable `PAPER_TEX` to the paper's `main.tex` to check that each one appears there verbatim.

In [15]:
# Smaller exponents beat the default by more than gt_cut % only in the tied design
gt_cut = np.ceil(max(gsum["close"]["e9"][1], gsum["spread"]["e9"][1]))
assert gsum["tied"]["e9"][0] > gt_cut
# Expected range of 10 i.i.d. standard normals (observed spread of 10 tied models, in standard errors)
tied_range10 = np.ptp(np.random.default_rng(0).standard_normal((200_000, 10)), axis=1).mean()
PAPER_TEX = os.environ.get("PAPER_TEX")
text = re.sub(r"\s+", " ", Path(PAPER_TEX).read_text()) if PAPER_TEX else None
au, V, LP = audit, A["swebench_verified"], A["livebench_pooled"]
vm, vb = V["models"], V["position_bias"].adjustment_pp.to_numpy()
pos1 = au.position_1_adjustment_pp
dv, dt = d.loc["swebench_verified"], d.loc["tau2_telecom"]
ps = [abs(d.loc[(k, "pooled"), "position_1_adjustment_pp"] - d.loc[(k, "stratified"), "position_1_adjustment_pp"]) for k in swe]
vs = per_position[per_position.setting == "swebench_verified"].set_index("position")
meth = [methods[x] for x in ["moon", "ordinary", "empirical_bayes"]]
s, l = toy["small_gap"], toy["large_gap"]
u, dd = toy06["unresolved"], toy06["decisive"]
G = gamma.set_index(["setting", "gamma"]).position_1_adjustment_pp
# Over 0.4 <= gamma <= 0.9: adjustments are positive, the default gives the smallest, and the adjustment keeps
# its order of magnitude except in Telecom and pooled LiveBench
GR = gamma_range.pivot(index="setting", columns="gamma", values="position_1_adjustment_pp")
assert (GR > 0).all().all() and (GR.idxmin(axis=1) == 0.9).all()
gr_ratio = GR.max(axis=1) / GR.min(axis=1)
assert set(gr_ratio[gr_ratio >= 4].index) == {"tau2_telecom", "livebench_pooled"}
raw_gap = vm.raw_score_pct[0] - vm.raw_score_pct[4]
slot_gap = raw_gap - vb[0] + vb[4]
# Margins by which the SWE-bench Verified top score rose in 2025 (same-day entries combined)
lb = read(REPO / "resources/swe_bench_history/verified/snapshots" / SWE_SNAPSHOT / "leaderboard.csv")
daily_top = lb.assign(date=pd.to_datetime(lb.date)).groupby("date").resolved.max()
record = daily_top.cummax()
gains = (record - record.shift()).loc[lambda x: (x > 0) & (x.index.year == 2025)]
claims = {
    "position-1 range": f"{pos1.min():.2f} to {pos1.max():.2f}",
    "Verified leader adjustment": f"adjustment of {vm.adjustment_pp[0]:.2f} percentage points",
    "Verified adjusted score": f"adjusted scores of {vm.raw_score_pct[0] - vm.adjustment_pp[0]:.1f}",
    "Verified record gains": f"rose {len(gains)} times. The median increase was {gains.median():.1f} percentage points, and "
                             f"{(gains < vm.adjustment_pp[0]).sum()} of the {len(gains)} increases were smaller",
    "Verified Tier 1": f"places {au.loc['swebench_verified', 'tier_1_size_80']} of {len(vm)} models",
    "Verified rank-1 mass": "{:.1f}\\% and {:.1f}\\% of the bootstrap rank-1 mass, while the model at rank 3 receives {:.1f}\\%".format(*(100 * V["rpm"][:3, 0])),
    "settings with > 1 pp": f"exceed 1 pp in {(au.leader_adjustment_pp > 1).sum()}",
    "settings with > 1 pp (intro)": f"exceed 1 percentage point in {(au.leader_adjustment_pp > 1).sum()} benchmarks",
    "settings with multi-model Tier 1": f"multiple models in {(au.tier_1_size_80 > 1).sum()} benchmarks",
    "Tau2 Airline": f"On Tau2 Airline, the leader's score falls by {au.loc['tau2_airline', 'leader_adjustment_pp']:.2f} pp after adjustment and "
                    f"{'all ' if au.loc['tau2_airline', 'tier_1_size_80'] == au.loc['tau2_airline', 'n_models'] else str(au.loc['tau2_airline', 'tier_1_size_80']) + ' of '}"
                    f"{au.loc['tau2_airline', 'n_models']} models share the leading tier",
    "LiveBench IF Tier 1": f"has the largest adjustment, {pos1['livebench_instruction_following']:.2f} pp, with "
                           f"{au.loc['livebench_instruction_following', 'tier_1_size_80']} of {au.loc['livebench_instruction_following', 'n_models']} models in the leading tier",
    "Verified Tier 1 size": f"groups {au.loc['swebench_verified', 'tier_1_size_80']} models into the leading tier",
    "Verified tiers": f"the full leaderboard has {au.loc['swebench_verified', 'tiers_80']} tiers",
    "Verified 90/95/99 Tier 1": f"expands the leading tier from {au.loc['swebench_verified', 'tier_1_size_80']} models to "
                                f"{au.loc['swebench_verified', 'tier_1_size_90']}, {au.loc['swebench_verified', 'tier_1_size_95']}, "
                                f"and {au.loc['swebench_verified', 'tier_1_size_99']}",
    "Verified leader intervals": f"rank intervals span positions {vm.rank_low_80[0]}--{vm.rank_high_80[0]}, and the rank-3 model's interval spans {vm.rank_low_80[2]}--{vm.rank_high_80[2]}",
    "Verified positions 9-10": f"[{vm.rank_low_80[8]},{vm.rank_high_80[8]}]",
    "LiveBench pooled leader": f"its leader has a {pos1['livebench_pooled']:.2f}-pp adjustment, {100 * LP['rpm'][0, 0]:.1f}\\% bootstrap rank-1 mass, and a "
                               f"{'singleton' if au.loc['livebench_pooled', 'tier_1_size_80'] == 1 else 'multi-model'} leading tier",
    "WildBench": f"WildBench's {au.loc['wildbench_v2', 'leader_adjustment_pp']:.2f}-pp adjustment and {au.loc['wildbench_v2', 'tier_1_size_80']}-model leading tier",
    "Verified 1-5 displayed": f"positions 1 and 5 consequently falls from {raw_gap:.2f} to {raw_gap - vm.adjustment_pp[0] + vm.adjustment_pp[4]:.2f} pp after "
                              f"tie-averaged adjustments of approximately {vm.adjustment_pp[0]:.2f} and {vm.adjustment_pp[4]:.2f} pp",
    "Verified 1-5 positions": f"gives a gap of {slot_gap:.2f} pp, a reduction of about {100 * (1 - slot_gap / raw_gap):.0f}\\%",
    "sizing delta": f"\\approx{delta:.1f}$ pp",
    "dependence Verified": f"{dv.loc['pooled', 'position_1_adjustment_pp']:.2f}, {dv.loc['stratified', 'position_1_adjustment_pp']:.2f}, and {dv.loc['wild', 'position_1_adjustment_pp']:.2f} pp",
    "dependence Verified Tier 1": f"it reduces the leading tier from {dv.loc['pooled', 'tier_1_size_80']:.0f} to {dv.loc['wild', 'tier_1_size_80']:.0f} models",
    "pooled vs stratified": f"differ by at most {max(ps):.2f} pp",
    "Telecom cluster": f"from {dt.loc['pooled', 'position_1_adjustment_pp']:.2f} to {dt.loc['wild', 'position_1_adjustment_pp']:.2f} pp and lowers the leader's rank-1 mass from "
                       f"{100 * dt.loc['pooled', 'p_rank_1_position_1']:.1f}\\% to {100 * dt.loc['wild', 'p_rank_1_position_1']:.1f}\\%",
    "split rank 1": f"from {agg.raw[1]:.2f} to {agg.adjusted[1]:.2f} pp",
    "split reduction rank 1": f"a reduction of {agg.reduction_pct[1]:.1f}\\%",
    "split reduction ranks 2-5": f"ranging from {agg.reduction_pct.loc[2:].min():.1f}\\% to {agg.reduction_pct.loc[2:].max():.1f}\\%",
    "Verified split rank 1": f"the mean winner error falls from {vs.split_error_pp[1]:.2f} to {vs.residual_pp[1]:.2f} pp",
    "calibrated (intro)": f"from {true_bias[0]:.2f} to {meth[0].residual_pp[1]:.2f} pp",
    "calibrated residuals": "leaves a mean winner residual of {:.2f} pp, compared with {:.2f} pp for the standard bootstrap and {:.2f} pp for EB".format(*[x.residual_pp[1] for x in meth]),
    "calibrated % removed": "approximately {:.0f}\\%, {:.0f}\\%, and {:.0f}\\%".format(*[x.bias_removed_pct[1] for x in meth]),
    "calibrated splitting residual": f"Sample splitting leaves a mean residual of {methods['sample_splitting'].residual_pp[1]:.2f} pp",
    "calibrated RMSE (main)": f"lowers the winner's RMSE from {100 * raw_rmse[0]:.2f} to {methods['moon'].rmse_pp[1]:.2f} pp",
    "calibrated RMSE": "has the lowest RMSE, {:.2f} pp, compared with {:.2f} pp for the standard bootstrap, {:.2f} pp for sample splitting, and {:.2f} pp for EB".format(
        *[methods[x].rmse_pp[1] for x in ["moon", "ordinary", "sample_splitting", "empirical_bayes"]]),
    "calibrated residual (main)": f"reduces it to {methods['moon'].residual_pp[1]:.2f} pp",
    "calibrated RMSE ranks 4-5": f"reduces the RMSE at positions 4 and 5 by about {raw_gap_pp[3]:.1f} pp",
    "calibrated adjustment SD": "standard deviation of {:.2f} pp across synthetic benchmarks, compared with {:.2f} pp".format(
        *[100 * reps[reps.position == 1][x].std() for x in ["sample_splitting", "moon"]]),
    "calibrated paired RMSE": "is {:.2f} pp (SE {:.2f}) for the standard bootstrap, {:.2f} pp (SE {:.2f}) for sample splitting, and {:.2f} pp (SE {:.2f}) for EB".format(
        *[v for x in ["ordinary", "sample_splitting", "empirical_bayes"] for v in (methods[x].rmse_gap_pp[1], methods[x].rmse_gap_se_pp[1])]),
    "calibrated paired EB rank 2": f"same RMSE to the displayed precision (SE of the paired difference {methods['empirical_bayes'].rmse_gap_se_pp[2]:.2f})",
    "calibrated RMSE spread": "RMSE differs by at most {:.2f} pp across methods, whereas mean residuals differ by up to {:.2f} pp".format(
        np.ptp([methods[x].rmse_pp[1] for x in METHODS]), np.ptp([methods[x].residual_pp[1] for x in METHODS])),
    "calibration cohort": f"fitted to the {int(calibration['n_eff'])} nonconstant rows of Verified",
    "calibration sigma": f"$\\hat\\sigma_T={float(calibration['sigma']):.4f}$",
    "toy small gap scores": f"averages {100 * s['raw_mean']:.2f}\\%, while the selected model's true ability averages {100 * s['selected_true_mean']:.2f}\\%",
    "toy small gap wins": f"The worse model wins in {100 * (1 - s['better_model_wins']):.1f}\\% of draws, and the better model receives only {100 * s['rpm'][1][0]:.1f}\\% rank-1 mass",
    "toy large gap mass": f"rank-1 mass rises to {100 * l['rpm'][1][0]:.1f}\\%",
    "toy small gap bias": f"Mean bias falls from {s['raw_bias_pp']:.2f} pp (Monte Carlo SE {s['raw_bias_se_pp']:.3f}) to a residual of {s['residual_pp']:.2f} pp (SE {s['residual_se_pp']:.3f})",
    "toy exact bias, small gap": f"population raw bias {s['exact_bias_pp']:.3f} pp",
    "toy exact bias, large gap": f"exact population winner bias falls to {l['exact_bias_pp']:.4f} pp",
    "toy large gap": f"${l['raw_bias_pp']:.3f}$ pp (SE {l['raw_bias_se_pp']:.3f})",
    "toy large gap adjustment": f"applies a mean adjustment of {l['mean_adjustment_pp']:.3f} pp, leaving a simulated mean residual of ${l['residual_pp']:.3f}$ pp (SE {l['residual_se_pp']:.3f})",
    "toy gamma 0.6 small gap": f"raw bias of {100 * u['bias1']:.3f} pp becomes a smaller residual of {100 * u['resid1']:.3f} pp",
    "toy gamma 0.6 large gap": f"Raw bias of {100 * dd['bias1']:.3f} pp then becomes a residual of ${100 * dd['resid1']:.3f}$ pp",
    "toy gamma 0.6 adjustment": f"rank-1 mass falls to {100 * dd['R_marg'][1][0]:.1f}\\%, compared with {100 * l['rpm'][1][0]:.1f}\\% at $m=\\lfloor N^{{0.9}}\\rfloor$, "
                                f"and the mean adjustment rises to approximately {100 * (dd['raw1'] - dd['corr1']):.3f} pp",
    "gamma table audited spans": "span {:.1f}--{:.1f} standard errors (median {:.1f})".format(spans["all"].min(), spans["all"].max(), spans["all"].median()),
    "gamma table top-10 spans": f"span {spans.top10.min():.1f}--{spans.top10.max():.1f} standard errors and lie within 3 standard errors in {(spans.top10 <= 3).sum()} benchmarks",
    "gamma table tied best": "lie between {:g} and {:g} and rise with $J$, reaching {:g}--{:g} at $J=100$".format(*gsum["tied"]["best"], *gsum["tied"]["best100"]),
    "gamma table tied default": "RMSE {:.0f}--{:.0f}\\% above that of the most accurate exponent for $J\\le10$ and {:.0f}--{:.0f}\\% above it at $J=100$".format(*gsum["tied"]["e9_10"], *gsum["tied"]["e9_100"]),
    "gamma table tied standard": "The standard bootstrap is {:.0f}--{:.0f}\\% above it.".format(*gsum["tied"]["e1"]),
    "gamma table close default": "within {:.0f}\\% of the most accurate exponent for $J\\ge20$ and within {:.0f}\\% for every $J$, while the standard bootstrap is {:.0f}--{:.0f}\\% above it".format(
        np.ceil(gsum["close"]["e9_20"][1]), np.ceil(gsum["close"]["e9"][1]), *gsum["close"]["e1"]),
    "gamma table spread best": "lie between {:g} and {:g}, and the default is within {:.0f}\\% of the best".format(*gsum["spread"]["best"], np.ceil(gsum["spread"]["e9"][1])),
    "gamma table default overall": "within {:.0f}\\% of the most accurate exponent in the close and spread designs".format(np.ceil(max(gsum["close"]["e9"][1], gsum["spread"]["e9"][1]))),
    "gamma table tied gain": "improve on the default by more than {:.0f}\\% only in the tied design, where the gain grows with $J$".format(gt_cut),
    "gamma table tied range": f"Even 10 tied models have an expected observed range of about {tied_range10:.0f} standard errors",
    "gamma range count": f"reports the {GR.size} adjustments",
    "gamma range close plateau": "each exponent from 0.4 to 0.9 has an RMSE within {:.0f}\\% of the most accurate exponent".format(np.ceil(ex["range"].xs("close", level="spread").max())),
    "gamma range separated leaders": "receive {:.1f}\\% and {:.1f}\\% of the bootstrap rank-1 mass at the default".format(*(100 * au.loc[["tau2_telecom", "livebench_pooled"], "leader_p_rank_1"])),
    "gamma table MC SE": f"at most {np.ceil(gt.rmse_se.max() * 1e4) / 100:.2f} pp",
}
assert round(methods["empirical_bayes"].rmse_pp[2], 2) == round(methods["moon"].rmse_pp[2], 2)
assert gt_overlap, "shaded exponents no longer overlap across N and p"
assert f"{raw_gap_pp[3]:.1f}" == f"{raw_gap_pp[4]:.1f}", "positions 4 and 5 no longer round to the same RMSE reduction"
check = pd.DataFrame({"expected text": claims})
if text is None:
    print(f"{len(check)} quoted numbers recomputed (set PAPER_TEX to check them against the paper source)")
else:
    check["found"] = [x in text for x in check["expected text"]]
    print(f"{check.found.sum()} of {len(check)} quoted numbers found in the paper")
check[~check.found] if text is not None and not check.found.all() else check

68 of 68 quoted numbers found in the paper


,expected text,found
position-1 range,0.09 to 3.86,True
Verified leader adjustment,adjustment of 0.90 percentage points,True
Verified adjusted score,adjusted scores of 78.3,True
Verified record gains,rose 13 times. The median increase was 0.8 per...,True
Verified Tier 1,places 13 of 173 models,True
...,...,...
gamma table tied range,Even 10 tied models have an expected observed ...,True
gamma range count,reports the 84 adjustments,True
gamma range close plateau,each exponent from 0.4 to 0.9 has an RMSE with...,True
gamma range separated leaders,receive 94.3\% and 99.0\% of the bootstrap ran...,True
